In [1]:
import nest_asyncio
nest_asyncio.apply()
import asyncio
from pathlib import Path

Path("hotel").mkdir(exist_ok=True)

async def step1_screenshot_hotels():
    import nodriver as uc
    browser = await uc.start(
        headless=True,
        browser_args=["--no-sandbox", "--disable-dev-shm-usage"],
    )
    page = await browser.get("https://www.google.com/travel/hotels?q=hotel%20%C4%90%C3%A0%20N%E1%BA%B5ng%20S%C6%A1n%20Tr%C3%A0")
    await asyncio.sleep(5)
    
    # Scroll xuống load nhiều hotel
    for i in range(5):
        await page.evaluate("window.scrollBy(0, 800)")
        await asyncio.sleep(2)
    
    # Scroll về đầu rồi chụp full page
    await page.evaluate("window.scrollTo(0, 0)")
    await asyncio.sleep(1)
    
    # Chụp full page
    await page.save_screenshot("hotel/step1_hotels.png", full_page=True)
    print("✅ Đã lưu: hotel/step1_hotels.png")
    
    browser.stop()

await step1_screenshot_hotels()

✅ Đã lưu: hotel/step1_hotels.png


In [1]:
import nest_asyncio
nest_asyncio.apply()
import asyncio
from pathlib import Path

Path("hotel").mkdir(exist_ok=True)

async def step1_booking_fullpage():
    import nodriver as uc
    browser = await uc.start(
        headless=False,
        browser_args=["--no-sandbox", "--disable-dev-shm-usage"],
    )
    
    page = await browser.get("https://www.booking.com/searchresults.html?ss=Đà+Nẵng")
    await asyncio.sleep(5)
    
    # Đóng modal
    await page.evaluate("""
        const closeBtns = document.querySelectorAll('button, [role="button"]');
        for (const btn of closeBtns) {
            const text = btn.textContent || btn.getAttribute('aria-label') || '';
            if (text.includes('Close') || text.includes('×') || text.includes('✕')) {
                btn.click(); break;
            }
        }
        const modals = document.querySelectorAll('[data-testid="modal"], [role="dialog"], .modal, [class*="modal"]');
        for (const m of modals) { m.style.display = 'none'; m.remove(); }
        const overlays = document.querySelectorAll('[data-testid="overlay"], [class*="overlay"]');
        for (const o of overlays) { o.style.display = 'none'; o.remove(); }
        const fixed = document.querySelectorAll('*');
        for (const el of fixed) {
            const style = window.getComputedStyle(el);
            if (style.position === 'fixed' && style.zIndex > 100) {
                el.style.display = 'none';
            }
        }
    """)
    await asyncio.sleep(2)
    
    # Scroll xuống cuối để load lazy content
    await page.evaluate("""
        async function scrollToBottom() {
            for (let i = 0; i < 10; i++) {
                window.scrollBy(0, 1000);
                await new Promise(r => setTimeout(r, 1000));
            }
        }
        scrollToBottom();
    """)
    await asyncio.sleep(12)
    
    # Scroll về đầu
    await page.evaluate("window.scrollTo(0, 0)")
    await asyncio.sleep(1)
    
    # Chụp 1 tấm full page
    await page.save_screenshot("hotel/step1_booking.png", full_page=True)
    print("✅ Đã lưu: hotel/step1_booking.png")
    
    browser.stop()

await step1_booking_fullpage()

✅ Đã lưu: hotel/step1_booking.png


In [7]:
import easyocr
import cv2
import time
import numpy as np
from pathlib import Path
from IPython.display import display, HTML

HOTEL_DIR = Path("hotel")

# Tìm tất cả ảnh trong thư mục hotel
image_paths = sorted(HOTEL_DIR.glob("*.png")) + sorted(HOTEL_DIR.glob("*.jpg"))
if not image_paths:
    raise FileNotFoundError(f"Không tìm thấy ảnh trong {HOTEL_DIR}")

print(f"📁 Tìm thấy {len(image_paths)} ảnh: {[p.name for p in image_paths]}")

# Khởi tạo EasyOCR
print("⏳ Khởi tạo EasyOCR...")
reader = easyocr.Reader(['vi', 'en'], gpu=False)

all_texts = []

for img_path in image_paths:
    print(f"\n🔍 Đang OCR: {img_path.name}")
    
    img = cv2.imread(str(img_path))
    if img is None:
        print(f"  ⚠️ Không đọc được ảnh")
        continue
    
    # Preprocess
    h, w = img.shape[:2]
    if w > 1440:
        scale = 1440 / w
        img = cv2.resize(img, (1440, int(h * scale)), interpolation=cv2.INTER_AREA)
    
    # OCR
    results = reader.readtext(img, detail=0, paragraph=False, batch_size=8)
    
    # Filter: chỉ giữ text dài > 10 ký tự, có chữ hoa (tên hotel)
    for text in results:
        text = text.strip()
        if len(text) > 10 and any(c.isupper() for c in text[1:]):
            all_texts.append(text)
            print(f"  ✓ {text[:60]}")
    
    print(f"  📊 {len(results)} dòng, giữ lại {len([t for t in results if len(t.strip()) > 10])}")

# Deduplicate
unique_texts = []
for t in all_texts:
    if t not in unique_texts:
        unique_texts.append(t)

print(f"\n{'='*60}")
print(f"✅ Tổng cộng: {len(unique_texts)} tên hotel unique")

# Lưu vào file
output_file = HOTEL_DIR / "competitors.txt"
output_file.write_text("\n".join(unique_texts), encoding="utf-8")
print(f"💾 Đã lưu: {output_file}")

# Hiển thị
print(f"\n{'='*60}")
print("DANH SÁCH:")
for i, name in enumerate(unique_texts, 1):
    print(f"{i}. {name}")

Using CPU. Note: This module is much faster with a GPU.


📁 Tìm thấy 2 ảnh: ['step1_booking.png', 'step1_hotels.png']
⏳ Khởi tạo EasyOCR...

🔍 Đang OCR: step1_booking.png
  ✓ Golden LBtusS Hoteieda Nang
  📊 5 dòng, giữ lại 3

🔍 Đang OCR: step1_hotels.png
  ✓ hotel Đà Nẫng S_
  ✓ Marriott Dan
  ✓ Four Points by Sheraton
  ✓ rriott Danan
  ✓ Four Points bv Sheraton Dan
  ✓ ~Star hotel
  ✓ Sơn Trà . 2,629 results
  ✓ Sontra Sea Hotel
  ✓ Ania Beachside Hotel Da
  ✓ LE SANDS OCEANFRONT DANANG
  ✓ Peninsula Hote
  ✓ Palmier Hotel
  ✓ GREAT PRICE
  ✓ ~Pectaurant
  ✓ Son Tra Resort & Spa
  ✓ 4-Starhotel
  ✓ Golden Sea Hotel
  ✓ The Code Hotel
  ✓ 1-Starhotel
  ✓ GREAT PRICE
  ✓ The Backpacker Hostel
  ✓ GREAT PRICE
  ✓ Nguyen Gia Hotel
  ✓ InterContinental Danang Sun Penin
  ✓ Breakfast (S)
  ✓ Hà Nội Hotel
  ✓ Mangata Beachfront Hotel
  ✓ OYO 853 Son Tra Motel
  ✓ Dolphin Hotel and Apartment
  ✓ SNite Hotel
  ✓ Dorm-Danang
  ✓ ALYSSA Da Nang Hotel
  ✓ VACATION RENTAL
  ✓ Banana Flower Homestav
  ✓ Studio with Bal .
  ✓ vour Internet address
  ✓ Hel

In [8]:
from groq import Groq
from pathlib import Path

# ─ Groq Setup ─
client = Groq(api_key="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")
MODEL = "meta-llama/llama-4-scout-17b-16e-instruct"

def call_groq(prompt: str, max_tokens: int = 1000) -> str:
    try:
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_completion_tokens=max_tokens,
            temperature=0.3,
        )
        return resp.choices[0].message.content or ""
    except Exception as e:
        print(f"⚠️ Groq error: {e}")
        return ""

# Đọc file OCR
raw_text = Path("hotel/competitors.txt").read_text(encoding="utf-8")

# Prompt LLM trích tên hotel
prompt = f"""Bạn là chuyên gia du lịch Đà Nẵng. Từ đoạn text OCR lộn xộn sau, hãy trích xuất CHÍNH XÁC tên các khách sạn, resort, homestay ở khu vực Sơn Trà/Đà Nẵng.

Yêu cầu:
- Chỉ trả về tên hotel, mỗi dòng 1 tên
- Loại bỏ: giá tiền, số sao, số reviews, tiện ích (WiFi, pool...), UI text
- Sửa lỗi OCR: "LBtusS" → "Lotus", "bv" → "by", v.v.
- Giữ nguyên tên thương hiệu: "Four Points by Sheraton", "InterContinental"

TEXT OCR:
{raw_text[:8000]}

OUTPUT (chỉ tên hotel, không giải thích):"""

print("⏳ Đang gọi Groq...")
result = call_groq(prompt, max_tokens=2000)

# Lưu kết quả
output_file = Path("hotel/competitors_clean.txt")
output_file.write_text(result, encoding="utf-8")

print(f"✅ Đã lưu: {output_file}")
print(f"📊 {len(result.splitlines())} hotel")
print("\n" + "="*60)
print(result)

⏳ Đang gọi Groq...
✅ Đã lưu: hotel/competitors_clean.txt
📊 22 hotel

Golden Lotus Hotel
Marriott Danang
Four Points by Sheraton
Sontra Sea Hotel
Ania Beachside Hotel
Le Sands Oceanfront Danang
Peninsula Hotel
Palmier Hotel
Son Tra Resort & Spa
Golden Sea Hotel
The Code Hotel
The Backpacker Hostel
Nguyen Gia Hotel
InterContinental Danang Sun Peninsula
Hà Nội Hotel
Mangata Beachfront Hotel
OYO 853 Son Tra Motel
Dolphin Hotel and Apartment
SNite Hotel
Dorm-Danang
ALYSSA Da Nang Hotel
Banana Flower Homestay


In [9]:
from ddgs import DDGS
from pathlib import Path
import json

# Đọc danh sách hotel
hotels = Path("hotel/competitors_clean.txt").read_text(encoding="utf-8").splitlines()
hotels = [h.strip() for h in hotels if h.strip()]
print(f"🔍 Tìm website cho {len(hotels)} hotel...")

ddgs = DDGS()
results = []

for i, name in enumerate(hotels, 1):
    print(f"\n{i}. {name}")
    
    try:
        # Tìm website chính thức
        search = ddgs.text(f"{name} official website Đà Nẵng", max_results=3)
        
        website = None
        for r in search:
            url = r['href']
            # Ưu tiên website chính thức, không phải booking/ota
            if any(x in url.lower() for x in ['booking.com', 'tripadvisor', 'agoda', 'expedia']):
                continue
            website = url
            break
        
        results.append({
            "name": name,
            "website": website,
            "search_results": [r['href'] for r in search[:3]]
        })
        
        print(f"   ✅ {website or 'Không tìm thấy'}")
        
    except Exception as e:
        print(f"   ❌ Lỗi: {e}")
        results.append({
            "name": name,
            "website": None,
            "search_results": []
        })

# Lưu JSON
output = Path("hotel/competitors_with_website.json")
output.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"\n{'='*60}")
print(f"✅ Đã lưu: {output}")
print(f"📊 {len([r for r in results if r['website']])}/{len(results)} có website")

🔍 Tìm website cho 22 hotel...

1. Golden Lotus Hotel
   ✅ https://goldenlotusgrandhotel.com/

2. Marriott Danang
   ✅ https://www.marriott.com/en-us/hotels/dadmr-danang-marriott-resort-and-spa/overview/

3. Four Points by Sheraton
   ✅ https://www.marriott.com/en-us/hotels/dadfp-four-points-danang/overview/

4. Sontra Sea Hotel
   ✅ https://sontra-sea.danang-hotels.com/en/

5. Ania Beachside Hotel
   ✅ https://haianbeachhotelspa.com/

6. Le Sands Oceanfront Danang
   ✅ https://lesandshotel.com/

7. Peninsula Hotel
   ✅ https://peninsulahotel.vn/en

8. Palmier Hotel
   ✅ https://palmierhoteldanang.vn-hotels.com/en/

9. Son Tra Resort & Spa
   ✅ https://sontraresort.com.vn/

10. Golden Sea Hotel
   ✅ https://reserving.com/hotels/asia/vietnam/da-nang/da-nang/golden-sea-hotel

11. The Code Hotel
   ✅ https://thecodehotel.com.vn/

12. The Backpacker Hostel
   ✅ https://www.hostelworld.com/hostels/p/320930/the-backpacker-hostel-and-spa/

13. Nguyen Gia Hotel
   ✅ https://nguyengiahotel.com/


In [11]:
import asyncio
import json
from pathlib import Path
from crawl4ai import AsyncWebCrawler, BrowserConfig, CrawlerRunConfig
from crawl4ai.content_filter_strategy import PruningContentFilter
from crawl4ai.markdown_generation_strategy import DefaultMarkdownGenerator

# Đọc danh sách
data = json.loads(Path("hotel/competitors_with_website.json").read_text(encoding="utf-8"))
hotels_with_site = [h for h in data if h["website"]]
print(f"🔍 Scrape {len(hotels_with_site)} website...")

async def scrape_hotel(url: str, name: str):
    try:
        browser_cfg = BrowserConfig(headless=True, extra_args=["--no-sandbox"])
        run_cfg = CrawlerRunConfig(
            page_timeout=15000,  # Giảm timeout
            markdown_generator=DefaultMarkdownGenerator(
                content_filter=PruningContentFilter(threshold=0.48)
            ),
        )
        async with AsyncWebCrawler(config=browser_cfg) as crawler:
            cr = await crawler.arun(url=url, config=run_cfg)
            text = cr.markdown.fit_markdown if hasattr(cr.markdown, "fit_markdown") else str(cr.markdown)
            return {
                "name": name,
                "url": url,
                "content": text[:3000] if text else "Empty",
                "status": cr.status_code,
                "success": len(text) > 100 if text else False
            }
    except Exception as e:
        return {
            "name": name,
            "url": url,
            "content": f"Error: {str(e)[:100]}",
            "status": 0,
            "success": False
        }

# Scrape tất cả
scraped = []
for i, hotel in enumerate(hotels_with_site, 1):
    print(f"\n{i}. {hotel['name']}")
    result = await scrape_hotel(hotel["website"], hotel["name"])
    scraped.append(result)
    
    status = "✅" if result["success"] else "⚠️"
    print(f"   {status} {len(result['content'])} ký tự | HTTP {result['status']}")

# Lưu
output = Path("hotel/competitors_scraped_all.json")
output.write_text(json.dumps(scraped, indent=2, ensure_ascii=False), encoding="utf-8")

success_count = sum(1 for s in scraped if s["success"])
print(f"\n{'='*60}")
print(f"✅ Đã lưu: {output}")
print(f"📊 {success_count}/{len(scraped)} website thành công")

🔍 Scrape 22 website...

1. Golden Lotus Hotel


[INIT].... → Crawl4AI 0.8.7 

[FETCH]... ↓ https://goldenlotusgrandhotel.com/                                                                   | ✓ | ⏱: 1.84s 

[SCRAPE].. ◆ https://goldenlotusgrandhotel.com/                                                                   | ✓ | ⏱: 0.00s 

[ERROR]... × https://goldenlotusgrandhotel.com/                 | Error: Blocked by anti-bot protection: Structural: no <body> tag (33056 bytes) 

...

[INIT].... → Crawl4AI 0.8.7 

[FETCH]... ↓ https://www.airbnb.com/rooms/640353223061005579                                                      | ✓ | ⏱: 1.56s 

[SCRAPE].. ◆ https://www.airbnb.com/rooms/640353223061005579                                                      | ✓ | ⏱: 0.08s 

[COMPLETE] ● https://www.airbnb.com/rooms/640353223061005579                                                      | ✓ | ⏱: 1.68s 

   ✅ 106 ký tự | HTTP 200

✅ Đã lưu: hotel/competitors_scraped_all.json
📊 19/22 website thành công


In [13]:
from groq import Groq
from pathlib import Path
import json

client = Groq(api_key="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")
MODEL = "meta-llama/llama-4-scout-17b-16e-instruct"

def call_groq(prompt: str, max_tokens: int = 2000) -> str:
    try:
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_completion_tokens=max_tokens,
            temperature=0.3,
        )
        return resp.choices[0].message.content or ""
    except Exception as e:
        print(f"⚠️ Groq error: {e}")
        return ""

# Đọc data
scraped = json.loads(Path("hotel/competitors_scraped_all.json").read_text(encoding="utf-8"))
success_hotels = [h for h in scraped if h["success"]]

competitor_info = "\n\n".join([
    f"--- {h['name']} ---\nURL: {h['url']}\n{h['content'][:1200]}"
    for h in success_hotels[:8]
])

prompt = f"""Phân tích cạnh tranh khách sạn Đà Nẵng cho Sontra Sea Hotel (3 sao, view biển Sơn Trà, 41 Hoàng Sa).

ĐỐI THỦ:
{competitor_info}

YÊU CẦU - Trả lời bằng tiếng Việt, định dạng đơn giản:

1. PHÂN LOẠI ĐỐI THỦ
   - Direct: Cùng phân khúc 2-3 sao, gần biển Sơn Trà
   - Indirect: Khác phân khúc hoặc xa khu vực

2. ĐỐI THỦ CHÍNH (Direct)
   Với mỗi đối thủ:
   - Tên
   - Điểm mạnh vs Sontra
   - Điểm yếu vs Sontra
   - Giá ước tính
   - Khách hàng mục tiêu

3. CHIẾN LƯỢC ĐỀ XUẤT
   - 3-5 hành động cụ thể cho Sontra Sea Hotel

KHÔNG dùng JSON. Viết dạng bullet point."""

print("⏳ Đang phân tích...")
result = call_groq(prompt, max_tokens=4000)

print("\n" + "="*60)
print("PHÂN TÍCH ĐỐI THỦ")
print("="*60)
print(result)

# Lưu plain text
Path("hotel/competitor_analysis.txt").write_text(result, encoding="utf-8")
print(f"\n💾 Đã lưu: hotel/competitor_analysis.txt")

⏳ Đang phân tích...

PHÂN TÍCH ĐỐI THỦ
1. **PHÂN LOẠI ĐỐI THỦ**
   - **Direct**: 
     - Ania Beachside Hotel (4 sao, gần biển, view tốt)
     - Le Sands Oceanfront Danang (4 sao, oceanfront, tiện ích cao cấp)
     - Peninsula Hotel (4 sao, luxury, view biển)
     - Son Tra Resort & Spa (4 sao, beachfront, khu vực yên tĩnh)
     - The Code Hotel & Spa (3-4 sao, seaside, tiện ích hiện đại)
   - **Indirect**:
     - Golden Sea Hotel (không rõ xếp hạng, khu vực khác)
     - Palmier Hotel (không rõ xếp hạng, khu vực khác)

2. **ĐỐI THỦ CHÍNH (Direct)**
   - **Ania Beachside Hotel**
     - **Điểm mạnh**: Tiện ích cao cấp, vị trí gần biển, phòng rộng rãi
     - **Điểm yếu**: Giá cao hơn, khách hàng mục tiêu là nhóm khách du lịch cao cấp
     - **Giá ước tính**: Cao hơn Sontra Sea Hotel
     - **Khách hàng mục tiêu**: Khách du lịch cao cấp, gia đình, cặp đôi
   - **Le Sands Oceanfront Danang**
     - **Điểm mạnh**: Tiện ích cao cấp, view oceanfront, phòng rộng rãi
     - **Điểm yếu**: Giá cao

In [ ]:
import requests
from pathlib import Path
import json

# ── 3.1: Google Trends ──
print("=" * 60)
print("BƯỚC 3: THU THẬP DỮ LIỆU KHÁCH HÀNG")
print("=" * 60)

# Google Trends API (unofficial) - pytrends
try:
    from pytrends.request import TrendReq
    pytrends = TrendReq(hl='vi-VN', tz=360)
    
    keywords = ["khách sạn giá rẻ Đà Nẵng", "hotel Sơn Trà", "du lịch Đà Nẵng bình dân"]
    pytrends.build_payload(keywords, timeframe='today 12-m', geo='VN')
    
    trends_data = pytrends.interest_over_time()
    print("\n📈 Google Trends:")
    print(trends_data.tail())
    
    # Lưu
    trends_data.to_csv("hotel/google_trends.csv")
    print("✅ Đã lưu: hotel/google_trends.csv")
    
except ImportError:
    print("⚠️ Chưa cài pytrends. Cài bằng: uv pip install pytrends")
    print("   Hoặc dùng cách khác...")

# ── 3.2: Reddit ──
print("\n🔍 Reddit...")
try:
    import praw
    
    reddit = praw.Reddit(
        client_id="YOUR_CLIENT_ID",
        client_secret="YOUR_SECRET",
        user_agent="hotel_research_bot"
    )
    
    subreddits = ["VietNam", "travel", "solotravel", "Shoestring"]
    reddit_posts = []
    
    for sub in subreddits:
        for post in reddit.subreddit(sub).search("Da Nang hotel", limit=10):
            reddit_posts.append({
                "title": post.title,
                "text": post.selftext[:500],
                "score": post.score,
                "comments": post.num_comments
            })
    
    print(f"✅ Tìm thấy {len(reddit_posts)} posts")
    
    # Lưu
    Path("hotel/reddit_posts.json").write_text(
        json.dumps(reddit_posts, indent=2, ensure_ascii=False), 
        encoding="utf-8"
    )
    print("✅ Đã lưu: hotel/reddit_posts.json")
    
except Exception as e:
    print(f"⚠️ Reddit lỗi: {e}")
    print("   Cần API key hoặc dùng pushshift.io")

# ── 3.3: Tổng hợp ──
print(f"\n{'='*60}")
print("TỔNG HỢP DỮ LIỆU")
print(f"{'='*60}")

# Đọc tất cả data đã thu thập
sources = []

if Path("hotel/google_trends.csv").exists():
    sources.append("Google Trends")
    
if Path("hotel/reddit_posts.json").exists():
    sources.append("Reddit")

print(f"✅ Nguồn dữ liệu: {', '.join(sources) if sources else 'Chưa có'}")
print("👉 Tiếp theo: LLM phân tích tạo personas")

In [19]:
import nest_asyncio
nest_asyncio.apply()
import asyncio
from pathlib import Path

Path("hotel").mkdir(exist_ok=True)

async def scrape_social_no_login():
    import nodriver as uc
    browser = await uc.start(
        headless=False,
        browser_args=["--no-sandbox", "--disable-dev-shm-usage"],
    )
    
    # 1. Facebook - search public posts
    print("🔍 Facebook...")
    page = await browser.get("https://www.facebook.com/search/posts?q=khách%20sạn%20Sơn%20Trà%20Đà%20Nẵng%20giá%20rẻ")
    await asyncio.sleep(8)
    await page.save_screenshot("hotel/facebook_search.png", full_page=True)
    print("✅ facebook_search.png")
    
    # 2. Instagram - hashtag
    print("🔍 Instagram...")
    page = await browser.get("https://www.instagram.com/explore/tags/dananghotel/")
    await asyncio.sleep(8)
    await page.save_screenshot("hotel/instagram_hashtag.png", full_page=True)
    print("✅ instagram_hashtag.png")
    
    # 3. TikTok - search
    print("🔍 TikTok...")
    page = await browser.get("https://www.tiktok.com/search?q=khách%20sạn%20Đà%20Nẵng%20giá%20rẻ")
    await asyncio.sleep(8)
    await page.save_screenshot("hotel/tiktok_search.png", full_page=True)
    print("✅ tiktok_search.png")
    
    # 4. Booking.com - reviews (đã thử, cần đóng modal)
    print("🔍 Booking.com reviews...")
    page = await browser.get("https://www.booking.com/hotel/vn/son-tra-sea.html#tab-reviews")
    await asyncio.sleep(5)
    
    # Đóng modal nếu có
    await page.evaluate("""
        document.querySelectorAll('[role="dialog"], [class*="modal"]').forEach(m => m.remove());
        document.querySelectorAll('[class*="overlay"]').forEach(o => o.remove());
    """)
    await asyncio.sleep(2)
    
    await page.save_screenshot("hotel/booking_reviews.png", full_page=True)
    print("✅ booking_reviews.png")
    
    browser.stop()
    print("\n✅ Tất cả đã chụp xong!")

await scrape_social_no_login()

✅ Clicked video 1
❌ Video 1 lỗi: "Tab" has no attribute "go_back"
✅ Clicked video 2
❌ Video 2 lỗi: "Tab" has no attribute "go_back"
✅ Clicked video 3
❌ Video 3 lỗi: "Tab" has no attribute "go_back"
✅ Clicked video 4
❌ Video 4 lỗi: "Tab" has no attribute "go_back"
✅ Clicked video 5
❌ Video 5 lỗi: "Tab" has no attribute "go_back"

✅ Đã lấy 0 videos


In [24]:
import nest_asyncio
nest_asyncio.apply()
import asyncio
from pathlib import Path

async def tiktok_inspect_html():
    import nodriver as uc
    browser = await uc.start(headless=False)
    
    page = await browser.get("https://www.tiktok.com/search?q=review%20khách%20sạn%20Đà%20Nẵng")
    await asyncio.sleep(10)
    
    # Scroll load nhiều video
    for i in range(5):
        await page.evaluate("window.scrollBy(0, 800)")
        await asyncio.sleep(2)
    
    # Lấy outerHTML của toàn bộ document (tương đương inspect → copy HTML)
    html = await page.evaluate("document.documentElement.outerHTML")
    
    # Lưu
    Path("hotel/tiktok_full_html.html").write_text(html, encoding="utf-8")
    print(f"✅ HTML: {len(html)} ký tự")
    print(f"💾 Saved: hotel/tiktok_full_html.html")
    
    browser.stop()

await tiktok_inspect_html()

✅ HTML: 577003 ký tự
💾 Saved: hotel/tiktok_full_html.html


In [25]:
from trafilatura import extract
from pathlib import Path
import json

html = Path("hotel/tiktok_full_html.html").read_text(encoding="utf-8")

# Dùng trafilatura trích nội dung chính
text = extract(
    html,
    output_format="json",
    include_comments=True,
    include_tables=False,
    deduplicate=True
)

print(f"✅ Trích xuất: {len(str(text))} ký tự")
print("=" * 60)
print(str(text)[:3000])

# Lưu
Path("hotel/tiktok_trafilatura.json").write_text(
    str(text) if text else "{}", 
    encoding="utf-8"
)
print(f"\n💾 Saved: hotel/tiktok_trafilatura.json")

✅ Trích xuất: 2312 ký tự
{"text": "Dưới đây là tổng hợp các khách sạn ở Đà Nẵng dành cho mấy ní thích view biển #danang #reviewdanang #khachsandanang #bienmykhedanang #dulich\nLên cho mn em khách sạn 3 sao mới keng - xịn đẹp tại Đà Nẵng #danang #dulichdanang #hotel #diff2026 #reviewdanang\ndulichdanang24\n5-26\nKhách sạn 4 sao đối diện biển Mỹ Khê với view cực đẹp, zá chỉ từ 6xx/người/đêm #reviewkhachsandanang #minhtoansafioceanhotel #khachsandepdanang #khachsan4saodanang\nreviewkhachsandanang43rv\n1w ago\nChỉ có 450k thì có búc được khách sạn view biển ở Đà Nẵng không ta?? #danang #dulichdanang #coemdanang #reviewdanang #khachsandanang\nLưu ngay list khách sạn giá mềm cho mùa du lịch hè ở Đà Nẵng\nmai.iu.danang\n5-13\n4 khách sạn view biển gần trung tâm chỉ từ 300k tại Đà Nẵng #dulichdanang #danang #dulich #reviewdanang #reviewdulich\ndulichdanang24\n3-31\nBà nào đang tìm khách sạn đẹp – giá hợp lý tại Đà Nẵng, mình đã tổng hợp một vài khách sạn ổn áp để mọi người dễ lựa chọn khi đi d

In [32]:
import nest_asyncio
nest_asyncio.apply()
import asyncio
import re
from pathlib import Path

async def tiktok_video_comments():
    import nodriver as uc
    browser = await uc.start(headless=False)
    
    # Vào trang search
    page = await browser.get("https://www.tiktok.com/search?q=review%20khách%20sạn%20Đà%20Nẵng")
    await asyncio.sleep(8)
    
    # Lấy HTML để tìm link video
    html = await page.get_content()
    video_links = list(set(re.findall(r'https://www\.tiktok\.com/@[\w.]+/video/\d+', html)))
    print(f"✅ Tìm thấy {len(video_links)} video links")
    
    # Vào từng video
    for i, link in enumerate(video_links[:3], 1):
        print(f"\n🔍 Video {i}: {link}")
        
        video_page = await browser.get(link)
        await asyncio.sleep(5)
        
        # Click tab "Comments" nếu có
        try:
            comments_tab = await video_page.find("Comments", best_match=True)
            await comments_tab.click()
            print("  ✅ Clicked Comments tab")
            await asyncio.sleep(3)
        except:
            print("  ⚠️ Không thấy Comments tab")
        
        # Kéo xuống load comment
        for scroll in range(5):
            await video_page.evaluate("window.scrollBy(0, 800)")
            await asyncio.sleep(2)
        
        # Lấy HTML
        video_html = await video_page.evaluate("document.documentElement.outerHTML")
        Path(f"hotel/tiktok_video_{i}_comments.html").write_text(video_html, encoding="utf-8")
        print(f"  💾 Saved: {len(video_html)} chars")
        
        # Chụp
        await video_page.save_screenshot(f"hotel/tiktok_video_{i}.png")
        print(f"  📸 Screenshot")
    
    browser.stop()
    print("\n✅ Xong!")

await tiktok_video_comments()

Video 1: 0 comments with username
Video 2: 0 comments with username
Video 3: 0 comments with username

✅ Tổng: 0 unique comments with username
💾 Saved: hotel/tiktok_comments_with_user.json

Mẫu:


In [35]:
from bs4 import BeautifulSoup
from pathlib import Path
import re
import json

all_comments = []

for i in range(1, 4):
    html_path = Path(f"hotel/tiktok_video_{i}_comments.html")
    if not html_path.exists():
        continue
    
    html = html_path.read_text(encoding="utf-8")
    soup = BeautifulSoup(html, "html.parser")
    
    comments = []
    
    # Tìm tất cả comment containers
    # TikTok dùng DivCommentContentWrapper hoặc data-e2e="comment-username-*"
    for container in soup.find_all("div", class_=re.compile(r"DivCommentContentWrapper|comment-content")):
        
        # Tìm username - từ thẻ p hoặc a trong comment header
        username_tag = container.find("p", class_=re.compile(r"TUXText.*weight-medium"))
        if not username_tag:
            username_tag = container.find("a", href=re.compile(r"/@"))
        
        username = username_tag.get_text(strip=True) if username_tag else "unknown"
        
        # Tìm text comment - từ thẻ p hoặc span trong content
        text_tags = container.find_all("p", class_=re.compile(r"TUXText"))
        # Lọc bỏ tag username, giữ lại tag text
        text = ""
        for tag in text_tags:
            tag_text = tag.get_text(strip=True)
            if tag_text != username and len(tag_text) > 5:
                text = tag_text
                break
        
        # Nếu không tìm được bằng class, thử tìm span có text dài
        if not text:
            spans = container.find_all("span")
            for span in spans:
                span_text = span.get_text(strip=True)
                if len(span_text) > 10 and span_text != username:
                    text = span_text
                    break
        
        # Filter
        if username and text and len(text) > 10:
            if any(k in text.lower() for k in ["khách sạn", "hotel", "đà nẵng", "phòng", "giá", "view", "biển", "sơn trà", "có", "không", "xin", "ib"]):
                comments.append({
                    "video": i,
                    "username": username,
                    "text": text,
                    "source": "tiktok_comment"
                })
    
    print(f"Video {i}: {len(comments)} comments with username")
    all_comments.extend(comments)

# Deduplicate
seen = set()
unique = []
for c in all_comments:
    key = c["username"] + c["text"][:30]
    if key not in seen:
        seen.add(key)
        unique.append(c)

print(f"\n{'='*60}")
print(f"✅ Tổng: {len(unique)} unique comments with username")

# Lưu
Path("hotel/tiktok_comments_with_user.json").write_text(
    json.dumps(unique, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print(f"💾 Saved: hotel/tiktok_comments_with_user.json")
print("\nMẫu:")
for i, c in enumerate(unique[:15], 1):
    print(f"\n{i}. @{c['username']} [Video {c['video']}]")
    print(f"   {c['text'][:120]}...")

Video 1: 0 comments with username
Video 2: 13 comments with username
Video 3: 17 comments with username

✅ Tổng: 30 unique comments with username
💾 Saved: hotel/tiktok_comments_with_user.json

Mẫu:

1. @XiMuoi [Video 2]
   ngày 20/6 có phòng 4 ng k shop...

2. @NHU PHAN [Video 2]
   xin giá 1/8 den 5/8...

3. @Chiến là Tên [Video 2]
   Xin giá 16-19/6 ạ...

4. @hong88 [Video 2]
   Xin giá 9-11/6 ạ...

5. @Thuy Luong6311 [Video 2]
   Ib phòng 2 ng...

6. @M🐱 [Video 2]
   𝗫𝗲 Đà 𝐍ẵ𝐧𝐠 - 𝐇ộ𝐢 𝐀𝐧 - 𝐁à 𝐍à - 𝐇𝐮ế
🌿Xe 4 chỗ giá 219k 1 chiều ( bao xe )
🌿Xe 7 chỗ giá 269k 1 chiều ( bao xe )
🌿Xe 16 chỗ...

7. @Ngọc Đạt [Video 2]
   xin giá ngày 2-6/6...

8. @Nguyễn Phương Thảo [Video 2]
   M xin giá phòng 5-8/6 2 giường...

9. @Diễm Kiều [Video 2]
   E xin zá gđ nhỏ 4 ng ạ...

10. @BẢO THƠM💗 HERA [Video 2]
   ib phòng 2 giường...

11. @Bộp [Video 2]
   Ib phòng 2 người...

12. @ʜ ᴜ ʏ ᴇ ɴ ᴛ ʀ ᴀ ɴ ɢ [Video 2]
   Rep ib e với ạ...

13. @Trần Thu Hiền [Video 2]
   Ib giúp mình ạ...

14. @Qn 🪐 [Video 3]


In [36]:
from groq import Groq
from pathlib import Path
import json

# ── Groq Setup ──
client = Groq(api_key="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")
MODEL = "meta-llama/llama-4-scout-17b-16e-instruct"

def call_groq(prompt: str, max_tokens: int = 4000) -> str:
    try:
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_completion_tokens=max_tokens,
            temperature=0.3,
        )
        return resp.choices[0].message.content or ""
    except Exception as e:
        print(f"⚠️ Groq error: {e}")
        return ""

# ── Đọc tất cả dữ liệu ──
competitors = Path("hotel/competitors_clean.txt").read_text(encoding="utf-8")
analysis = Path("hotel/competitor_analysis.txt").read_text(encoding="utf-8")
tiktok = json.loads(Path("hotel/tiktok_comments_with_user.json").read_text(encoding="utf-8"))

# Format TikTok comments
comments_text = "\n".join([
    f"@{c['username']}: {c['text'][:100]}"
    for c in tiktok[:30]
])

# ── Prompt tổng hợp ──
prompt = f"""Bạn là Giám đốc Marketing khách sạn Sontra Sea Hotel (3 sao, view biển Sơn Trà, 41 Hoàng Sa, Đà Nẵng).

DỮ LIỆU ĐÃ THU THẬP:

1. ĐỐI THỦ CHÍNH:
{competitors[:1500]}

2. PHÂN TÍCH ĐỐI THỦ:
{analysis[:2000]}

3. KHÁCH HÀNG THẬT (TikTok comments):
{comments_text[:2500]}

YÊU CẦU - Tạo báo cáo chiến lược:

A. CUSTOMER PERSONAS (3-5 personas)
   Mỗi persona gồm:
   - Tên & mô tả ngắn
   - Demographics (tuổi, thu nhập, nguồn gốc)
   - Pain points (điều họ ghét/lo lắng)
   - Motivations (điều họ tìm kiếm)
   - Kênh tiếp cận (TikTok, Facebook, Google, OTA...)

B. CHIẾN LƯỢC TIẾP CẬN TỪNG PERSONA
   - Message chính
   - Offer đề xuất
   - Kênh quảng cáo
   - Content format (video, ảnh, blog...)

C. ĐỀ XUẤT CHIẾN DỊCH 30 NGÀY
   - Tuần 1-2: Awareness
   - Tuần 3-4: Conversion

OUTPUT: Viết tiếng Việt, dạng bullet point, dễ đọc."""

print("⏳ Đang phân tích tổng hợp...")
result = call_groq(prompt, max_tokens=4000)

print("\n" + "=" * 70)
print("BÁO CÁO CHIẾN LƯỢC - SONTRA SEA HOTEL")
print("=" * 70)
print(result)

# Lưu
Path("hotel/final_strategy_report.txt").write_text(result, encoding="utf-8")
print(f"\n💾 Saved: hotel/final_strategy_report.txt")

⏳ Đang phân tích tổng hợp...

BÁO CÁO CHIẾN LƯỢC - SONTRA SEA HOTEL
### Báo cáo Chiến lược Marketing cho Sontra Sea Hotel

## A. Customer Personas

Dựa trên dữ liệu thu thập, chúng tôi xác định 3 customer personas chính cho Sontra Sea Hotel:

1. **Persona 1: Du khách gia đình - "Bố mẹ và bé"**
   - **Mô tả ngắn**: Gia đình trẻ với con nhỏ, tìm kiếm nơi nghỉ dưỡng thoải mái và tiện nghi.
   - **Demographics**: 
     - Tuổi: 25-45
     - Thu nhập: Trung bình đến cao (20-50 triệu/tháng)
     - Nguồn gốc: Hà Nội, TP.HCM và các tỉnh thành khác
   - **Pain points**: 
     - Lo lắng về sự an toàn và tiện nghi cho trẻ nhỏ.
     - Khó khăn trong việc tìm nơi phù hợp với gia đình.
   - **Motivations**:
     - Tìm kiếm không gian rộng rãi và thoải mái cho gia đình.
     - Tiện ích dành cho trẻ em và dịch vụ tốt.
   - **Kênh tiếp cận**: Facebook, Google, OTA (Agoda, Booking.com)

2. **Persona 2: Cặp đôi trẻ - "Tình yêu biển"**
   - **Mô tả ngắn**: Cặp đôi trẻ tuổi, yêu thích du lịch và khám phá.
 

In [37]:
from groq import Groq
from pathlib import Path
import json

client = Groq(api_key="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")
MODEL = "meta-llama/llama-4-scout-17b-16e-instruct"

def call_groq(prompt: str, max_tokens: int = 2000) -> str:
    try:
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_completion_tokens=max_tokens,
            temperature=0.3,
        )
        return resp.choices[0].message.content or ""
    except Exception as e:
        return f"Error: {e}"

# Đọc comments
comments = json.loads(Path("hotel/tiktok_comments_with_user.json").read_text(encoding="utf-8"))

# Chỉ lấy comments có nội dung hỏi giá/phòng (high intent)
high_intent = [c for c in comments if any(k in c["text"].lower() for k in ["xin giá", "có phòng", "ib phòng", "giá", "phòng"])]

print(f"🔍 Phân tích {len(high_intent)} khách hàng high-intent...")

results = []

for i, c in enumerate(high_intent[:15], 1):  # Giới hạn 15 để tiết kiệm token
    username = c["username"]
    text = c["text"]
    
    prompt = f"""Phân tích nhân khẩu học và hành vi từ TikTok user:

Username: @{username}
Comment: "{text}"

Yêu cầu phân tích:
1. Đoán giới tính (nam/nữ/khó xác định) và lý do
2. Đoán độ tuổi (Gen Z 18-25, Millennial 26-35, Gen X 36-50) và lý do
3. Đoán khu vực (Hà Nội, TP.HCM, miền Trung, khác) nếu có thể
4. Sở thích từ username và nội dung
5. Nhu cầu cụ thể (đi khi nào, bao nhiêu người, mục đích)
6. Khả năng có nhu cầu thật (cao/trung bình/thấp) và lý do

Trả lời ngắn gọn, bullet point."""

    analysis = call_groq(prompt, max_tokens=800)
    
    results.append({
        "username": username,
        "comment": text,
        "analysis": analysis
    })
    
    print(f"\n{'='*60}")
    print(f"@{username}")
    print(f"Comment: {text[:80]}...")
    print(f"\nPhân tích:")
    print(analysis)

# Lưu
Path("hotel/customer_deep_analysis.json").write_text(
    json.dumps(results, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print(f"\n{'='*60}")
print(f"✅ Đã phân tích {len(results)} khách hàng")
print(f"💾 Saved: hotel/customer_deep_analysis.json")

🔍 Phân tích 25 khách hàng high-intent...

@XiMuoi
Comment: ngày 20/6 có phòng 4 ng k shop...

Phân tích:
Dưới đây là phân tích nhân khẩu học và hành vi của người dùng TikTok @XiMuoi:

* **Giới tính:** Nữ (khó xác định chính xác nhưng có thể nghiêng về nữ do cách sử dụng ngôn ngữ và nội dung có thể liên quan đến đi shop)
* **Độ tuổi:** Gen Z (18-25) 
 * Lý do: Sử dụng ngôn ngữ và cách viết ngắn gọn, phổ biến ở giới trẻ (ngày 20/6 có phòng 4 ng k shop).
* **Khu vực:** Khó xác định (không có thông tin cụ thể về địa điểm)
* **Sở thích:** 
 * Có thể thích đi du lịch, khám phá (nhìn từ việc hỏi phòng)
 * Thích mua sắm (nhìn từ "k shop")
* **Nhu cầu cụ thể:** 
 * Đi vào ngày 20/6
 * Đi với 4 người
 * Mục đích có thể là du lịch, nghỉ dưỡng và mua sắm
* **Khả năng có nhu cầu thật:** Cao 
 * Lý do: Người dùng đã cụ thể hóa ngày và số lượng người, thể hiện sự quan tâm và sẵn sàng cho một kế hoạch cụ thể.

@NHU PHAN
Comment: xin giá 1/8 den 5/8...

Phân tích:
Dưới đây là phân tích nhân khẩu học và